In [ ]:
# Copyright 2026 Google LLC
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#     https://www.apache.org/licenses/LICENSE-2.0
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Module 09 — Prompt Caching

Cache a **large, reused prefix** so repeated calls are **cheaper and faster** — you mark the prefix with **`cache_control`** and the platform reuses it on later calls.

This builds on **Modules 00–08** and uses the same **`AnthropicVertex` / ADC** path.

## Part B — Bootstrap

Setup mirrors Module 00 — see that module for full detail (prerequisites, ADC, logging).

In [ ]:
%pip install --quiet "anthropic[vertex]"

In [ ]:
# ===== CONDENSED BOOTSTRAP (mirrors Module 00) =====
import subprocess
from anthropic import AnthropicVertex

def _default_project() -> str:
    try:
        out = subprocess.run(
            ["gcloud", "config", "get-value", "project"],
            capture_output=True, text=True, timeout=15,
        ).stdout.strip()
        if out and out != "(unset)":
            return out
    except Exception:
        pass
    return "<YOUR_PROJECT_ID>"

PROJECT_ID = _default_project()   # or hard-code: PROJECT_ID = "my-project-id"
LOCATION   = "global"
MODEL      = "claude-opus-4-8"

assert PROJECT_ID and PROJECT_ID != "<YOUR_PROJECT_ID>", (
    "PROJECT_ID is still the placeholder. Run `gcloud config set project <id>` "
    "so it auto-detects, or edit PROJECT_ID directly."
)

client = AnthropicVertex(project_id=PROJECT_ID, region=LOCATION)
print(f"✅ Client ready (project={PROJECT_ID}, region={LOCATION}, model={MODEL}).")

## Part C — Cache a large system prefix

Mark a reusable prefix with a **cache breakpoint**: add `"cache_control": {"type": "ephemeral"}` to the **last content block** of the prefix you want cached. It works on a **system-prompt block**, a **message content block**, or **tool definitions**.

Key facts:

- The cached prefix is reused on later calls — default **~5-minute TTL** (a **1-hour TTL** is available via a beta header).
- There's a **minimum cacheable prefix size** (roughly **~1024 tokens** for Opus-class models); below it, caching simply won't engage. Confirm exact minimums in the docs.
- `usage` reports **`cache_creation_input_tokens`** (written on the first call) and **`cache_read_input_tokens`** (read on cache hits), alongside `input_tokens` / `output_tokens`.
- Basic ephemeral caching uses the **regular `client.messages.create`** — no beta header needed.

Below we build a large, **synthetic, fictional** "operations handbook" so the prefix clearly exceeds the minimum.

In [ ]:
# Build a LARGE, self-contained, fictional system prefix (no copyrighted text).
sections = []
for i in range(1, 41):
    sections.append(
        f"Section {i}. Operations Handbook (synthetic, fictional content for caching demo). "
        f"All teams in region block {i} must record deployments in the central ledger before "
        f"rollout, validate health checks across every replica, and confirm rollback steps with "
        f"the on-call engineer. Capacity reviews occur each cycle; incident notes are filed within "
        f"one business day. Procedure {i}A covers staged rollouts, procedure {i}B covers emergency "
        f"freezes, and procedure {i}C covers post-incident review and sign-off responsibilities. "
        f"This text is intentionally repetitive padding to exceed the minimum cacheable size."
    )
big_text = (
    "You are an operations assistant. Use ONLY the handbook below to answer.\n\n"
    + "\n\n".join(sections)
)
print(f"Approx characters in system prefix: {len(big_text):,} (~{len(big_text)//4:,} rough tokens)")

resp1 = client.messages.create(
    model=MODEL,
    max_tokens=256,
    system=[{"type": "text", "text": big_text, "cache_control": {"type": "ephemeral"}}],
    messages=[{"role": "user", "content": "In one sentence: when must deployments be recorded?"}],
)

u1 = resp1.usage
print("\n--- First call usage ---")
print(f"input_tokens               : {u1.input_tokens}")
print(f"output_tokens              : {u1.output_tokens}")
print(f"cache_creation_input_tokens: {getattr(u1, 'cache_creation_input_tokens', None)}  (expect > 0)")
print(f"cache_read_input_tokens    : {getattr(u1, 'cache_read_input_tokens', None)}")

## Part D — Cache hit on the second call

A second call that reuses the **same cached system prefix** should **read from cache** — even with a different user question. Watch `cache_read_input_tokens` go above 0 and `input_tokens` drop.

In [ ]:
resp2 = client.messages.create(
    model=MODEL,
    max_tokens=256,
    system=[{"type": "text", "text": big_text, "cache_control": {"type": "ephemeral"}}],
    messages=[{"role": "user", "content": "In one sentence: who confirms rollback steps?"}],
)

u2 = resp2.usage
print("--- Second call usage ---")
print(f"input_tokens               : {u2.input_tokens}")
print(f"output_tokens              : {u2.output_tokens}")
print(f"cache_creation_input_tokens: {getattr(u2, 'cache_creation_input_tokens', None)}")
print(f"cache_read_input_tokens    : {getattr(u2, 'cache_read_input_tokens', None)}  (expect > 0)")
print("\nℹ️  If cache_read is 0, the prefix may be under the minimum cacheable size,")
print("    or the cache TTL (~5 min by default) lapsed between calls — re-run promptly.")

## Part E — Notes

- **Cache large, STABLE prefixes:** long system instructions, big shared context/documents, and tool definitions — content that doesn't change call to call.
- **Breakpoint placement:** put `cache_control` on the **last block** of the prefix you want cached.
- **Prefix-based:** caching matches a prefix, so **order matters** — keep stable content first and variable content (the user's question) after it.
- **TTL:** default **~5 minutes**; a **1-hour** TTL is available via a beta header.
- **Economics:** the first **write** costs slightly more than a normal input; subsequent **reads** cost much less — the win grows with prefix size and reuse frequency.
- Confirm minimums and current details in the docs: https://docs.claude.com/en/docs/build-with-claude/prompt-caching and https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/partner-models/claude/prompt-caching

## Part F — Recap

- Mark a reusable prefix with `"cache_control": {"type": "ephemeral"}` on its **last block** (system, message, or tools).
- The **first call writes** the cache (`cache_creation_input_tokens` > 0); **later calls read** it (`cache_read_input_tokens` > 0) and cost less.
- There's a **minimum cacheable size** (~1024 tokens, Opus-class) and a **~5-min default TTL** (1-hour via beta header) — confirm in docs.
- Basic ephemeral caching uses the **regular `client.messages.create`** — no beta header needed.

**Next: Module 10 — Count tokens.**